In [10]:
!python DTUeval-python/eval.py --data results_dtu_eval/pointcloud_to_mesh.ply --scan 24 --mode mesh --dataset_dir /home/ahc/Datasets/dtu/dtu_eval_data --vis_out_dir ./results_dtu_eval

done: 100%|███████████████████████████████████████| 9/9 [00:50<00:00,  5.62s/it]
3.592193558437384 2.1029768563027105 2.847585207370047


In [ ]:
!python DTUeval-python/eval.py --data results_dtu_eval/vggt_aligned.ply --scan 24 --mode pcd --dataset_dir /home/ahc/Datasets/dtu/dtu_eval_data --vis_out_dir ./results_dtu_eval

In [ ]:
from pathlib import Path
import copy
import numpy as np
import open3d as o3d


def _load_as_point_cloud(ply_path, sample_mesh_points=200_000):
    ply_path = Path(ply_path)
    mesh = o3d.io.read_triangle_mesh(str(ply_path))
    if not mesh.is_empty() and len(mesh.triangles) > 0:
        return mesh.sample_points_uniformly(number_of_points=sample_mesh_points)
    pcd = o3d.io.read_point_cloud(str(ply_path))
    if pcd.is_empty():
        raise RuntimeError(f"Could not load geometry from: {ply_path}")
    return pcd


def _bbox_diag(pcd):
    pts = np.asarray(pcd.points)
    return float(np.linalg.norm(pts.max(axis=0) - pts.min(axis=0)))


def _prescale_transform(pcd_a, pcd_b):
    """Build T that maps A's centroid+scale to match B's."""
    pts_a = np.asarray(pcd_a.points)
    pts_b = np.asarray(pcd_b.points)
    s = _bbox_diag(pcd_b) / _bbox_diag(pcd_a)
    mu_a = pts_a.mean(0)
    mu_b = pts_b.mean(0)
    T = np.eye(4)
    T[:3, :3] = s * np.eye(3)
    T[:3, 3] = mu_b - s * mu_a
    return T, s


def _fpfh(pcd, voxel):
    down = pcd.voxel_down_sample(voxel)
    down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel * 2, max_nn=30))
    feat = o3d.pipelines.registration.compute_fpfh_feature(
        down, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel * 5, max_nn=100)
    )
    return down, feat


def align_similarity_icp(
    ply_a,
    ply_b,
    output_overlay_ply,
    output_aligned_ply,
    output_aligned_only_ply,
    voxel_fraction=0.01,
    color_a=(1.0, 0.2, 0.2),
    color_b=(0.1, 0.6, 1.0),
):
    """
    Align A to B via:
      1. Centroid + bbox-diagonal pre-scale (removes the ~530x gap)
      2. FPFH RANSAC global registration (finds rotation)
      3. ICP with TransformationEstimationPointToPoint(with_scaling=True)

    output_overlay_ply:      raw A (unaligned) + B
    output_aligned_ply:      transformed A + B
    output_aligned_only_ply: transformed A only (optional)
    """
    pcd_a = _load_as_point_cloud(ply_a)
    pcd_b = _load_as_point_cloud(ply_b)

    diag_b = _bbox_diag(pcd_b)
    voxel = diag_b * voxel_fraction
    print(f"Diag A: {_bbox_diag(pcd_a):.4f}, B: {diag_b:.4f} | Voxel (B-scale): {voxel:.4f}")

    # Step 1: pre-scale A into B's coordinate frame
    T_pre, s_pre = _prescale_transform(pcd_a, pcd_b)
    pcd_a_scaled = copy.deepcopy(pcd_a)
    pcd_a_scaled.transform(T_pre)
    print(f"Pre-scale factor: {s_pre:.4f}")

    # Step 2: FPFH RANSAC (both clouds now at same scale)
    a_down, a_fpfh = _fpfh(pcd_a_scaled, voxel)
    b_down, b_fpfh = _fpfh(pcd_b, voxel)

    max_corr = voxel * 10.0
    print("Running FPFH RANSAC...")
    result_ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        a_down, b_down, a_fpfh, b_fpfh,
        mutual_filter=True,
        max_correspondence_distance=max_corr,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=4,
        checkers=[
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(max_corr),
        ],
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(200000, 0.999),
    )
    print(f"RANSAC fitness: {result_ransac.fitness:.4f}  rmse: {result_ransac.inlier_rmse:.4f}")

    # Step 3: ICP with isotropic scale
    print("Running similarity ICP...")
    result_icp = o3d.pipelines.registration.registration_icp(
        a_down, b_down,
        max_correspondence_distance=voxel * 2.0,
        init=result_ransac.transformation,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(with_scaling=True),
        criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=200),
    )
    print(f"ICP fitness: {result_icp.fitness:.4f}  rmse: {result_icp.inlier_rmse:.4f}")

    # Compose: T_final maps original A → B
    T_final = result_icp.transformation @ T_pre

    Path(output_overlay_ply).parent.mkdir(parents=True, exist_ok=True)

    # Raw overlay: unaligned A + B
    a_raw = copy.deepcopy(pcd_a)
    a_raw.paint_uniform_color(list(color_a))
    b_vis = copy.deepcopy(pcd_b)
    b_vis.paint_uniform_color(list(color_b))
    o3d.io.write_point_cloud(output_overlay_ply, a_raw + b_vis)
    print(f"Saved raw overlay:     {output_overlay_ply}")

    # Aligned overlay: transformed A + B
    a_aligned = copy.deepcopy(pcd_a)
    a_aligned.transform(T_final)
    a_aligned.paint_uniform_color(list(color_a))
    o3d.io.write_point_cloud(output_aligned_ply, a_aligned + b_vis)
    print(f"Saved aligned overlay: {output_aligned_ply}")

    # Aligned only: transformed A without B
    a_aligned_only = copy.deepcopy(pcd_a)
    a_aligned_only.transform(T_final)
    o3d.io.write_point_cloud(output_aligned_only_ply, a_aligned_only)
    print(f"Saved aligned only:    {output_aligned_only_ply}")

    return {"transform": T_final, "fitness": result_icp.fitness, "rmse": result_icp.inlier_rmse}


out = align_similarity_icp(
    ply_a="image_vggt_pointcloud.ply",
    ply_b="/home/ahc/Datasets/dtu/dtu_eval_data/Points/stl/stl024_total.ply",
    output_overlay_ply="results_dtu_eval/overlay_raw_A_plus_B_scan024.ply",
    output_aligned_ply="results_dtu_eval/overlay_aligned_A_to_B_scan024.ply",
    output_aligned_only_ply="results_dtu_eval/vggt_aligned.ply",
    voxel_fraction=0.01,
)
out

[Open3D WARNING] geometry::TriangleMesh appears to be a geometry::PointCloud (only contains vertices, but no triangles).
[Open3D WARNING] geometry::TriangleMesh appears to be a geometry::PointCloud (only contains vertices, but no triangles).
Diag A: 0.9585, B: 1088.2022 | Voxel (B-scale): 10.8820
Pre-scale factor: 1135.3620
Running FPFH RANSAC...
[Open3D WARNING] Too few correspondences (417) after mutual filter, fall back to original correspondences.
RANSAC fitness: 0.0000  rmse: 0.0000
Running similarity ICP...
ICP fitness: 0.8995  rmse: 6.1303
Saved raw overlay:     results_dtu_eval/overlay_raw_A_plus_B_scan024.ply
Saved aligned overlay: results_dtu_eval/overlay_aligned_A_to_B_scan024.ply
Saved aligned only:    results_dtu_eval/vggt_aligned.ply


{'transform': array([[ 661.69052362,  -14.65562996, -175.54056714,  194.18188073],
        [   7.20542973,  684.04294379,  -29.94929191,    6.51795334],
        [ 176.00386354,   27.09411032,  661.1748456 ,   32.73649206],
        [   0.        ,    0.        ,    0.        ,    1.        ]]),
 'fitness': 0.8995396696452749,
 'rmse': 6.13033722643263}

In [ ]:
# for raw scale
cam_large = np.load("/home/ahc/Datasets/dtu/public_data/dtu/dtu_scan24/cameras_large.npz", allow_pickle=True)
# for normalized scale (VGGT predicts the point maps in normalized (no actual normalization happens))
cam_sphere = np.load("/home/ahc/Datasets/dtu/public_data/dtu/dtu_scan24/cameras_sphere.npz", allow_pickle=True)
print(cam_large.files)
print(cam_sphere.files)

['scale_mat_0', 'scale_mat_inv_0', 'world_mat_0', 'world_mat_inv_0', 'camera_mat_0', 'camera_mat_inv_0', 'scale_mat_1', 'scale_mat_inv_1', 'world_mat_1', 'world_mat_inv_1', 'camera_mat_1', 'camera_mat_inv_1', 'scale_mat_2', 'scale_mat_inv_2', 'world_mat_2', 'world_mat_inv_2', 'camera_mat_2', 'camera_mat_inv_2', 'scale_mat_3', 'scale_mat_inv_3', 'world_mat_3', 'world_mat_inv_3', 'camera_mat_3', 'camera_mat_inv_3', 'scale_mat_4', 'scale_mat_inv_4', 'world_mat_4', 'world_mat_inv_4', 'camera_mat_4', 'camera_mat_inv_4', 'scale_mat_5', 'scale_mat_inv_5', 'world_mat_5', 'world_mat_inv_5', 'camera_mat_5', 'camera_mat_inv_5', 'scale_mat_6', 'scale_mat_inv_6', 'world_mat_6', 'world_mat_inv_6', 'camera_mat_6', 'camera_mat_inv_6', 'scale_mat_7', 'scale_mat_inv_7', 'world_mat_7', 'world_mat_inv_7', 'camera_mat_7', 'camera_mat_inv_7', 'scale_mat_8', 'scale_mat_inv_8', 'world_mat_8', 'world_mat_inv_8', 'camera_mat_8', 'camera_mat_inv_8', 'scale_mat_9', 'scale_mat_inv_9', 'world_mat_9', 'world_mat_inv

In [ ]:
scale_mat = cam_sphere['scale_mat_0']
scale_mat_inv = cam_sphere['scale_mat_inv_0']
world_mat = cam_sphere['world_mat_0']
world_mat_inv = cam_sphere['world_mat_inv_0']
camera_mat = cam_sphere['camera_mat_0']
camera_mat_inv = cam_sphere['camera_mat_inv_0']

In [31]:
for k in cam_large.files:
    if '_0' not in k:
        continue
    print(f"cam_large[{k}].shape: {cam_large[k].shape} dtype: {cam_large[k].dtype}")
    print(f"\t{cam_large[k]}")

print("-----")

for k in cam_sphere.files:
    if '_0' not in k:
        continue
    print(f"cam_sphere[{k}].shape: {cam_sphere[k].shape} dtype: {cam_sphere[k].dtype}")
    print(f"\t{cam_sphere[k]}")

cam_large[scale_mat_0].shape: (4, 4) dtype: float32
	[[669.99963    0.         0.        43.391827]
 [  0.       669.99963    0.       -18.58792 ]
 [  0.         0.       669.99963  613.8096  ]
 [  0.         0.         0.         1.      ]]
cam_large[scale_mat_inv_0].shape: (4, 4) dtype: float32
	[[ 0.00149254  0.          0.         -0.06476396]
 [ 0.          0.00149254  0.          0.02774318]
 [ 0.          0.          0.00149254 -0.9161342 ]
 [ 0.          0.          0.          1.        ]]
cam_large[world_mat_0].shape: (4, 4) dtype: float32
	[[ 2.6074299e+03 -3.8448980e+00  1.4981781e+03 -5.3393669e+05]
 [-1.9207690e+02  2.8625525e+03  6.8179816e+02  2.3434688e+04]
 [-2.4160500e-01 -3.0951001e-02  9.6988100e-01  2.2540121e+01]
 [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  1.0000000e+00]]
cam_large[world_mat_inv_0].shape: (4, 4) dtype: float32
	[[ 3.3546050e-04 -5.1133857e-06 -5.1459229e-01  1.9083348e+02]
 [ 2.5861191e-06  3.4666393e-04 -2.4768946e-01 -1.1601868e+00]
 [ 8.36